# 세계 언어 지수(Worldwide Language Index) — 최종 코퍼스 10,020건 재현 노트북

r22 노트북은 10개 언어·`ln(10)` 분모로 `fandom_scores_v6.json`(r22)의 language_coverage를 재검증했다. 최종 데이터는 14개 언어다.

| 입력 | 내용 |
|---|---|
| `fandom_scores_live_reference_v7.json` | 라이브 10,020건 coverage_detail(14개 언어 language_counts, 분모 ln(14)) |
| `worldwide_language_pilot_live_reference_v7.json` / `worldwide_language_index_v7.csv` | 팬덤별 세계 언어 지수 원본과 CSV |
| `language_domain_summary_v7.json` | 14개 언어별 근거 건수·도메인 수(코퍼스 합계) |
| `fandom_scores_v6.json` | 동결 스냅샷 7,350건(13개 언어, 아랍어 없음, 분모 ln(13)) — 비교용 |
| `archive/.../fandom_scores_v6.json` | r22 5,612건(10개 언어, ln(10)) — 비교용 |

In [1]:
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 140)


def find_repo_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "data" / "v7_final" / "fandoms_v3_100.json").exists():
            return p
    raise FileNotFoundError("저장소 루트(data/v7_final/fandoms_v3_100.json)를 찾지 못함 — 저장소 안에서 실행하세요")


REPO = find_repo_root()
DATA_DIR = REPO / "data" / "v7_final"                       # 최종 산출물(10,020건 라이브 + 동결 스냅샷 7,350건)
ROUNDS_DIR = REPO / "data" / "v7_rounds"                    # 병합 로그 r1~r72
ARCHIVE_DIR = REPO / "archive" / "v6_r22_era" / "data" / "v6_r22_snapshot"   # r22(5,612건) 비교용, 읽기 전용


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def flatten_bullets(fandoms):
    rows = []
    for rec in fandoms:
        for kind in ("loyalty", "spillover"):
            for item in rec.get(kind, []):
                rows.append({"fandom": rec["fandom"], "category": rec.get("category"),
                             "bullet_type": kind, "text": item.get("t", "") or "", "url": item.get("u", "") or ""})
    return pd.DataFrame(rows)

live_scores = load_json(DATA_DIR / "fandom_scores_live_reference_v7.json")
wl = load_json(DATA_DIR / "worldwide_language_pilot_live_reference_v7.json")
wl_csv = pd.read_csv(DATA_DIR / "worldwide_language_index_v7.csv", encoding="utf-8-sig")
lang_dom = load_json(DATA_DIR / "language_domain_summary_v7.json")
frozen_scores = load_json(DATA_DIR / "fandom_scores_v6.json")
r22_scores = load_json(ARCHIVE_DIR / "fandom_scores_v6.json")

ALL_LANGS = ["ko", "en", "ja", "zh", "es", "fr", "th", "id", "vi", "ru", "tl", "pt", "tr", "ar"]
LANG_LABEL = {"ko": "한국어", "en": "영어", "ja": "일본어", "zh": "중국어", "es": "스페인어", "fr": "프랑스어", "th": "태국어",
              "id": "인도네시아어", "vi": "베트남어", "ru": "러시아어", "tl": "필리핀어", "pt": "포르투갈어", "tr": "튀르키예어", "ar": "아랍어"}
langs_present = set()
for rec in live_scores:
    langs_present.update(rec["coverage_detail"]["language_counts"].keys())
print("라이브 점수에 등장하는 언어:", sorted(langs_present), "| 14개 스키마 안에 있음:", langs_present <= set(ALL_LANGS), "| 개수:", len(langs_present))
print("worldwide JSON 팬덤 수:", len(wl), "| CSV 행:", len(wl_csv), "| language_domain_summary 언어 수:", len(lang_dom))

라이브 점수에 등장하는 언어: ['ar', 'en', 'es', 'fr', 'id', 'ja', 'ko', 'pt', 'ru', 'th', 'tl', 'tr', 'vi', 'zh'] | 14개 스키마 안에 있음: True | 개수: 14
worldwide JSON 팬덤 수: 100 | CSV 행: 100 | language_domain_summary 언어 수: 14


## 1. LanguageCoverage 공식 재검증 — 세 시점 각각의 분모(ln(14) / ln(13) / ln(10))

In [2]:
def shannon_diversity(counts, denom_langs):
    total = sum(counts.values())
    if total == 0 or denom_langs <= 1:
        return 0.0
    ent = -sum((c / total) * math.log(c / total) for c in counts.values() if c > 0)
    return ent / math.log(denom_langs)


def verify(scores, denom, label):
    sum_mis = cov_mis = 0
    for rec in scores:
        cd = rec["coverage_detail"]
        if sum(cd["language_counts"].values()) != cd["n_evidence"]: sum_mis += 1
        if abs(shannon_diversity(cd["language_counts"], denom) - cd["language_coverage"]) > 0.001: cov_mis += 1
    langs = set(); [langs.update(r["coverage_detail"]["language_counts"]) for r in scores]
    print(f"[{label}] 언어 수 {len(langs)}, 분모 ln({denom}): 언어합!=n_evidence {sum_mis}/{len(scores)}, language_coverage 불일치 {cov_mis}/{len(scores)}")

verify(live_scores, 14, "라이브 10,020건")
verify(frozen_scores, 13, "동결 7,350건")
verify(r22_scores, 10, "r22 5,612건")
print("(다른 분모를 쓰면 100/100 불일치 — verify_v7_final_consistency.py [V]·[Z]와 동일한 결론)")

[라이브 10,020건] 언어 수 14, 분모 ln(14): 언어합!=n_evidence 0/100, language_coverage 불일치 0/100
[동결 7,350건] 언어 수 13, 분모 ln(13): 언어합!=n_evidence 0/100, language_coverage 불일치 0/100
[r22 5,612건] 언어 수 10, 분모 ln(10): 언어합!=n_evidence 0/100, language_coverage 불일치 0/100
(다른 분모를 쓰면 100/100 불일치 — verify_v7_final_consistency.py [V]·[Z]와 동일한 결론)


## 2. 세계 언어 지수 재계산 — 원본 스크립트 컬럼 스키마(14개 언어) 적용, JSON·CSV와 대조

In [3]:
FOREIGN = [l for l in ALL_LANGS if l != "ko"]
rows = []; mism = {"n_languages_hit": 0, "language_diversity": 0, "foreign_bullets": 0, "foreign_ratio": 0,
                   "n_foreign_languages_hit": 0, "foreign_diversity": 0, "primary_foreign_language": 0, "primary_foreign_share": 0}
for rec in live_scores:
    cd = rec["coverage_detail"]
    counts = {l: cd["language_counts"].get(l, 0) for l in ALL_LANGS}
    total = sum(counts.values()); fc = {l: counts[l] for l in FOREIGN}; ft = sum(fc.values())
    prim = max(fc, key=fc.get) if ft else None
    calc = {"n_languages_hit": sum(1 for c in counts.values() if c > 0), "language_diversity": round(shannon_diversity(counts, 14), 3),
            "foreign_bullets": ft, "foreign_ratio": round(ft / total, 3) if total else 0.0,
            "n_foreign_languages_hit": sum(1 for c in fc.values() if c > 0), "foreign_diversity": round(shannon_diversity(fc, 13), 3),
            "primary_foreign_language": prim, "primary_foreign_share": round(fc[prim] / ft, 3) if ft else 0.0}
    orig = wl[rec["fandom"]]
    for k, v in calc.items():
        o = orig[k]
        if (isinstance(v, float) and abs(v - o) > 0.0015) or (not isinstance(v, float) and v != o):
            mism[k] += 1
    row = {"팬덤": rec["fandom"], "근거문장수": total, "검출언어수": calc["n_languages_hit"], "언어다양성": calc["language_diversity"],
           "해외근거문장수": ft, "해외비중": calc["foreign_ratio"], "검출해외언어수": calc["n_foreign_languages_hit"],
           "해외언어다양성": calc["foreign_diversity"], "대표해외언어": LANG_LABEL.get(prim, ""), "대표해외언어비중": calc["primary_foreign_share"]}
    row.update({LANG_LABEL[l]: counts[l] for l in ALL_LANGS}); rows.append(row)
pilot_df = pd.DataFrame(rows).sort_values(["해외근거문장수", "팬덤"], ascending=[False, True]).reset_index(drop=True)
print("원본 JSON 대비 필드별 불일치 팬덤 수:", mism)
merged = pilot_df.merge(wl_csv, on="팬덤", suffixes=("", "_csv"))
csv_mis = {c: int((merged[c] != merged[c + "_csv"]).sum()) for c in ["근거문장수", "해외근거문장수", "검출언어수", "검출해외언어수"]}
print("CSV(worldwide_language_index_v7.csv) 대비 정수 컬럼 불일치:", csv_mis)
bts = pilot_df[pilot_df["팬덤"] == "BTS"].iloc[0]
print(f"BTS: 해외 {int(bts['해외근거문장수'])}건 ({bts['해외비중']:.1%}), 해외언어다양성 {bts['해외언어다양성']} (KEY_FINDINGS: 135건 60%, 0.66)")
pilot_df.index += 1
pilot_df.head(15)

원본 JSON 대비 필드별 불일치 팬덤 수: {'n_languages_hit': 0, 'language_diversity': 0, 'foreign_bullets': 0, 'foreign_ratio': 0, 'n_foreign_languages_hit': 0, 'foreign_diversity': 0, 'primary_foreign_language': 0, 'primary_foreign_share': 0}
CSV(worldwide_language_index_v7.csv) 대비 정수 컬럼 불일치: {'근거문장수': 0, '해외근거문장수': 0, '검출언어수': 0, '검출해외언어수': 0}
BTS: 해외 135건 (59.5%), 해외언어다양성 0.66 (KEY_FINDINGS: 135건 60%, 0.66)


,팬덤,근거문장수,검출언어수,언어다양성,해외근거문장수,해외비중,검출해외언어수,해외언어다양성,대표해외언어,대표해외언어비중,...,스페인어,프랑스어,태국어,인도네시아어,베트남어,러시아어,필리핀어,포르투갈어,튀르키예어,아랍어
1,BLACKPINK,195,14,0.752,135,0.692,13,0.770,영어,0.407,...,15,1,14,4,7,4,3,3,2,2
2,BTS,227,13,0.638,135,0.595,12,0.660,영어,0.526,...,16,4,0,5,1,4,3,6,3,2
3,Stray Kids,163,13,0.658,129,0.791,12,0.603,영어,0.589,...,11,1,4,8,0,9,3,4,1,1
4,TWICE,191,13,0.666,116,0.607,12,0.698,영어,0.379,...,9,0,4,2,1,2,3,4,1,1
5,SEVENTEEN,157,13,0.573,111,0.707,12,0.500,영어,0.685,...,5,1,1,3,1,0,3,2,1,2
6,NewJeans,154,13,0.634,108,0.701,12,0.591,영어,0.593,...,9,3,2,0,7,1,2,3,1,1
7,NCT,164,14,0.685,101,0.616,13,0.722,영어,0.426,...,4,1,7,4,2,1,4,3,1,1
8,LE SSERAFIM,143,13,0.700,97,0.678,12,0.701,영어,0.402,...,9,1,2,6,0,3,2,2,1,2
9,aespa,158,12,0.603,79,0.500,11,0.701,영어,0.456,...,10,0,3,3,2,0,2,3,1,2
10,ATEEZ,113,12,0.606,78,0.690,11,0.554,영어,0.628,...,8,1,4,2,0,1,3,3,1,1


## 3. 코퍼스 전체 언어 구성 — 세 시점 비교 + language_domain_summary 대조

In [4]:
def totals(scores):
    t = {}
    for rec in scores:
        for l, c in rec["coverage_detail"]["language_counts"].items():
            t[l] = t.get(l, 0) + c
    return t
live_t, frozen_t, r22_t = totals(live_scores), totals(frozen_scores), totals(r22_scores)
ld = {r["lang_code"]: r for r in lang_dom}
comp = pd.DataFrame([{"언어": LANG_LABEL[l], "code": l, "라이브 10,020": live_t.get(l, 0), "language_domain_summary": ld[l]["total_bullets"] if l in ld else None,
                      "도메인 수(라이브)": ld[l]["total_domains"] if l in ld else None, "동결 7,350": frozen_t.get(l, 0), "r22 5,612": r22_t.get(l, 0)}
                     for l in ALL_LANGS]).sort_values("라이브 10,020", ascending=False).reset_index(drop=True)
comp["라이브 비중"] = (comp["라이브 10,020"] / comp["라이브 10,020"].sum()).round(4)
print("합계: 라이브", comp["라이브 10,020"].sum(), "| summary", comp["language_domain_summary"].sum(), "| 동결", comp["동결 7,350"].sum(), "| r22", comp["r22 5,612"].sum())
print("라이브 language_counts 합 == language_domain_summary total_bullets (언어별 전부):", all(comp["라이브 10,020"] == comp["language_domain_summary"]))
comp

합계: 라이브 10020 | summary 10020 | 동결 7350 | r22 5612
라이브 language_counts 합 == language_domain_summary total_bullets (언어별 전부): True


,언어,code,"라이브 10,020",language_domain_summary,도메인 수(라이브),"동결 7,350","r22 5,612",라이브 비중
0,한국어,ko,5551,5551,704,4183,2956,0.5540
1,영어,en,2195,2195,79,1751,1561,0.2191
2,일본어,ja,520,520,86,333,304,0.0519
3,중국어,zh,458,458,67,153,152,0.0457
4,스페인어,es,250,250,63,121,121,0.0250
5,인도네시아어,id,210,210,16,193,192,0.0210
6,태국어,th,209,209,21,181,181,0.0209
7,필리핀어,tl,137,137,8,133,0,0.0137
8,프랑스어,fr,101,101,32,4,4,0.0101
9,포르투갈어,pt,89,89,18,87,0,0.0089


## 4. 한계

1. 언어는 출처 도메인 기준(`language_of()`)이며 본문 언어가 아니다 — 한국 매체의 영문 기사, 해외 매체의 한국어판 등은 도메인 언어로 분류된다(TOKENIZER 문서 참고).
2. 14개 언어 이외의 언어(예: 독일어·이탈리아어 매체)는 도메인 규칙에 없으면 기본값으로 흡수되므로 "검출언어수"는 하한이다.
3. 세 시점(r22 10개·동결 13개·라이브 14개)은 분모가 달라 language_coverage 절대치는 시점 간 비교할 수 없다.